# mRMR, IC ranking, Lasso and elastic net (Prompt #9, Steps 13-15, 20-23, 29-36)

Each method orders the in-fold candidates of one target; the first k features of each ordering are fitted on training rows (ridge) and scored on the next chronological block. mRMR (relevance |rank IC| or copula MI, redundancy |Spearman|, difference or quotient, lambda 0.5 / 1) is compared with the plain IC ranking, which piles up near-duplicates.

**This notebook contains no calculations.** Everything is computed by `xauusd_quant.selection` and `xauusd_quant.research.feature_selection_*` and read back from `results/feature_selection/`. Generate those first (needs the Prompt #8 research of the same timeframe for the pipeline-null veto):

```
xq feature-select --timeframe 5m                        # every stage + report + manifests
xq feature-select --timeframe 5m --target residual_reduction
xq feature-select --timeframe 5m --target future_return --horizon 5
xq feature-selection-report --timeframe 5m              # summary, SEL-H ledger rows, plots
xq build-feature-manifest --set standard --timeframe 5m # verify + reproduce the matrix
```

**Periods.** Development 2003-2017 selects; validation 2018-2021 only evaluates what development chose and reads the plateaus; the reserved test period (2022-) is never loaded with outcomes. Diagnostic models are linear (ridge, Lasso, elastic net, L1 logistic) on training-CDF rank designs - not the ML system of Prompt #10, and nothing here is a trading rule.

In [ ]:
import sys
import json
from pathlib import Path

# Work against the repository's own source tree, not an installed copy.
ROOT = Path.cwd().parent if Path.cwd().name == "research" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import polars as pl
from IPython.display import Image, display

from xauusd_quant.selection.config import load_selection_config

scfg = load_selection_config(ROOT / "config" / "feature_selection.yaml")

TIMEFRAME = "5m"       # 5m | 15m (configured); 30m | 1h if they were run

out = scfg.results_path / TIMEFRAME
if not out.exists():
    print(f"No selection for {TIMEFRAME}. Run:  xq feature-select --timeframe {TIMEFRAME}")

def table(name):
    """Load a previously generated selection table (None if it was not written)."""
    path = out / f"{name}.parquet"
    return pl.read_parquet(path) if path.exists() else None

def plot(name):
    path = out / "plots" / f"{name}.png"
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print(f"(no plot {path.name})")

def load_json(name):
    path = out / name
    return json.loads(path.read_text()) if path.exists() else {}

summary = load_json("summary.json")
sets = load_json("sets.json")
{k: v for k, v in summary.items() if not isinstance(v, (dict, list))}

## Nested learning curves by method

In [ ]:
plot('method_curves')

## mRMR vs the IC ranking (top 20, development -> validation)

In [ ]:
pl.DataFrame(summary.get('mrmr_vs_ic_ranking', []))

## mRMR orderings

In [ ]:
table('mrmr_results').filter((pl.col('split') == 'development_to_validation') & (pl.col('rank') <= 10))

## Lasso and elastic-net entry order, refit coefficients

In [ ]:
table('lasso_selection').filter(pl.col('entry_rank') <= 10).sort('target', 'split', 'entry_rank')

In [ ]:
summary.get('l1_consistent')

## Coefficient signs across folds

In [ ]:
plot('coefficient_stability')
table('coefficient_stability').filter(pl.col('sign_flips')).sort('target')

## Binary direction: L1 logistic path

In [ ]:
table('logistic_l1')